# 1. Limpieza de datos — Titanic

**Objetivo:** dejar el conjunto de datos del Titanic listo para modelos de aprendizaje supervisado.

En este cuaderno:
1. Cargamos los datos y revisamos su estructura.
2. Detectamos valores faltantes, duplicados y tipos incorrectos.
3. Imputamos faltantes con criterios justificados.
4. Revisamos valores atípicos.
5. Guardamos `data/titanic_clean.csv` para los siguientes cuadernos.

> Código generado con apoyo de IA generativa (Claude) y revisado paso a paso.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df = pd.read_csv("data/titanic.csv")
print("Filas, columnas:", df.shape)
df.head()

Filas, columnas: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1.1 Diccionario de datos

| Columna | Descripción |
|---|---|
| `PassengerId` | Identificador del pasajero |
| `Survived` | **Variable objetivo**: 0 = no sobrevivió, 1 = sobrevivió |
| `Pclass` | Clase del boleto (1 = primera, 2 = segunda, 3 = tercera) |
| `Name` | Nombre (incluye el título: Mr, Mrs, Miss…) |
| `Sex` | Sexo |
| `Age` | Edad en años |
| `SibSp` | Hermanos / cónyuges a bordo |
| `Parch` | Padres / hijos a bordo |
| `Ticket` | Número de boleto |
| `Fare` | Tarifa pagada |
| `Cabin` | Número de cabina |
| `Embarked` | Puerto de embarque (C = Cherbourg, Q = Queenstown, S = Southampton) |

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Braund, Mr. Owen Harris",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


## 1.2 Valores faltantes

In [4]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
}).sort_values("faltantes", ascending=False)
faltantes[faltantes["faltantes"] > 0]

,faltantes,porcentaje
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


**Lectura:**
- `Cabin` tiene ~77 % de faltantes: no se puede imputar con confianza, pero **el hecho de tener cabina registrada** es informativo (suele asociarse a primera clase). Extraeremos la **cubierta** (primera letra) y una bandera `HasCabin`.
- `Age` tiene ~20 % de faltantes: imputaremos con la **mediana por título** (Mr, Mrs, Miss, Master…), que es mucho más precisa que una mediana global (ej. "Master" son niños).
- `Embarked` tiene 2 faltantes: imputamos con la **moda**.

## 1.3 Duplicados y consistencia

In [5]:
print("Filas duplicadas:", df.duplicated().sum())
print("PassengerId duplicados:", df["PassengerId"].duplicated().sum())
print("Valores de Sex:", df["Sex"].unique())
print("Valores de Embarked:", df["Embarked"].unique())
print("Valores de Pclass:", sorted(df["Pclass"].unique()))
print("Valores de Survived:", sorted(df["Survived"].unique()))

Filas duplicadas: 0
PassengerId duplicados: 0
Valores de Sex: <StringArray>
['male', 'female']
Length: 2, dtype: str
Valores de Embarked: <StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str
Valores de Pclass: [np.int64(1), np.int64(2), np.int64(3)]
Valores de Survived: [np.int64(0), np.int64(1)]


## 1.4 Extraer el título del nombre

In [6]:
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [7]:
mapa_titulos = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "the Countess": "Rare", "Countess": "Rare", "Dona": "Rare",
    "Capt": "Rare", "Col": "Rare", "Don": "Rare", "Dr": "Rare",
    "Major": "Rare", "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare",
}
df["Title"] = df["Title"].replace(mapa_titulos)
df["Title"].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

## 1.5 Imputación de `Age` por título

In [8]:
mediana_edad_titulo = df.groupby("Title")["Age"].median()
print(mediana_edad_titulo)

df["AgeWasMissing"] = df["Age"].isna().astype(int)
df["Age"] = df["Age"].fillna(df["Title"].map(mediana_edad_titulo))
print("\nFaltantes en Age después de imputar:", df["Age"].isna().sum())

Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      48.5
Name: Age, dtype: float64

Faltantes en Age después de imputar: 0


## 1.6 Imputación de `Embarked` y tratamiento de `Cabin`

In [9]:
moda_embarked = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(moda_embarked)
print("Moda de Embarked:", moda_embarked)

df["Deck"] = df["Cabin"].str[0].fillna("U")      # U = desconocida
df["Deck"] = df["Deck"].replace({"T": "U"})        # 'T' es un solo caso
df["HasCabin"] = (df["Deck"] != "U").astype(int)
df["Deck"].value_counts()

Moda de Embarked: S


Deck
U    688
C     59
B     47
D     33
E     32
A     15
F     13
G      4
Name: count, dtype: int64

## 1.7 Valores atípicos

In [10]:
for col in ["Age", "Fare", "SibSp", "Parch"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    fuera = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col:6s}  atípicos por IQR: {fuera:4d}   min={df[col].min():.2f}  max={df[col].max():.2f}")

Age     atípicos por IQR:   35   min=0.42  max=80.00
Fare    atípicos por IQR:  116   min=0.00  max=512.33
SibSp   atípicos por IQR:   46   min=0.00  max=8.00
Parch   atípicos por IQR:  213   min=0.00  max=6.00


**Decisión:** los atípicos de `Fare` (hasta ~512) y de `SibSp`/`Parch` son **reales** (boletos de lujo y familias numerosas), no errores de captura. Se **conservan**; los modelos de árboles son robustos a ellos y para los modelos lineales escalaremos las variables.

## 1.8 Tipos de datos y guardado

In [11]:
df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")
df["Pclass"] = df["Pclass"].astype(int)

columnas_finales = ["PassengerId", "Survived", "Pclass", "Name", "Title", "Sex", "Age",
                    "AgeWasMissing", "SibSp", "Parch", "Ticket", "Fare",
                    "Deck", "HasCabin", "Embarked"]
df_clean = df[columnas_finales].copy()

print("Faltantes totales:", df_clean.isna().sum().sum())
df_clean.to_csv("data/titanic_clean.csv", index=False)
print("Guardado data/titanic_clean.csv ->", df_clean.shape)
df_clean.head()

Faltantes totales: 0
Guardado data/titanic_clean.csv -> (891, 15)


,PassengerId,Survived,Pclass,Name,Title,Sex,Age,AgeWasMissing,SibSp,Parch,Ticket,Fare,Deck,HasCabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Mr,male,22.0,0,1,0,A/5 21171,7.2500,U,0,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,female,38.0,0,1,0,PC 17599,71.2833,C,1,C
2,3,1,3,"Heikkinen, Miss. Laina",Miss,female,26.0,0,0,0,STON/O2. 3101282,7.9250,U,0,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,female,35.0,0,1,0,113803,53.1000,C,1,S
4,5,0,3,"Allen, Mr. William Henry",Mr,male,35.0,0,0,0,373450,8.0500,U,0,S


## Resumen
- 891 pasajeros, sin duplicados.
- `Age` imputada por mediana de título (+ bandera `AgeWasMissing`).
- `Embarked` imputada con la moda.
- `Cabin` transformada en `Deck` y `HasCabin`.
- Nueva variable `Title` extraída del nombre.
- Atípicos conservados por ser reales.

**Siguiente paso:** `2_EDA.ipynb`.